# SLEAP → DLCAnalyzer formatting

Merges the `animal` and `geom` coordinate exports for one assay/cohort and writes
DLCAnalyzer-readable CSVs into `formatted/<PHASE>/`.

Run the cells in order. Configuration is in one place; nothing below it needs editing
when you move to another cohort or assay.

---

## What changed, and why

The previous version derived output names by counting underscores:

```python
new_file_name = file_name.split("_", 7)[-1] + "_formatted.csv"
```

On `B6_A3R3_S1.merged_locs` that splits into `['B6','A3R3','S1.merged','locs']` and takes
`'locs'`, so **every file was written as `locs_formatted.csv` and overwrote the last one**.
The count was right for a longer filename pattern used at some point and silently wrong
afterwards. Position-based string surgery on filenames breaks the moment the naming
changes, and it fails quietly.

Fixes in this version:

| Problem | Fix |
|---|---|
| `split("_", 7)` → all files named `locs_formatted.csv` | Parse `<CODE>_<PHASE>` with a regex; no underscore counting |
| Silent overwriting destroyed 19 files into 1 | Refuse to overwrite; abort on a name collision |
| Output written flat, then moved by hand | Written straight into `formatted/<PHASE>/` |
| Per-assay magic numbers (`+20` SocP, `+18` OFT, `+26` EPM) | Bodypart count read from the header |
| `animal`/`geom` paired by `str.replace` | Paired on the parsed `(code, phase)` key, with unmatched files reported |
| Row counts never checked | Asserted equal before merging |
| Hard-coded test path in the merge cell | One config cell for both steps |

The output format is unchanged: 3 header rows, then `frame, x, y, likelihood` per
bodypart, with `likelihood = 1` throughout because SLEAP's `analysis_locs` export carries
no confidence column. Existing formatted files are reproduced byte-for-byte.

## 1. Configuration

In [ ]:
from pathlib import Path
import re
import pandas as pd

# ---- edit this block only -------------------------------------------------
BEHAVIOR_ROOT = Path(r"S:\Lab_Member\Tobi\Experiments\Exp9_Social-Stress\Raw Data\Behavior")
BATCH = "B6"
ASSAY = "SocP"          # SocP | NOR | EPM | OFT
OVERWRITE = False       # True only when you intend to replace existing output
# ---------------------------------------------------------------------------

# Phases per assay. An empty tuple means the assay has a single unphased
# recording per animal, so files are named <CODE> and land flat in formatted/.
PHASES = {
    "SocP": ("HAB", "S1", "S2"),
    "NOR":  ("HAB", "NOV"),
    "EPM":  (),
    "OFT":  (),
}

# CRLF matches the existing formatted corpus, so re-running this notebook
# reproduces those files byte-for-byte instead of differing by line ending.
# R's read.csv and DLCAnalyzer accept either; the point is consistency.
LINE_TERM = "\r\n"

SLEAP     = BEHAVIOR_ROOT / BATCH / ASSAY / "SLEAP"
ANIMAL    = SLEAP / "animal"
GEOM      = SLEAP / "geom"
MERGED    = SLEAP / "merged"      # was the SLEAP root; its own folder keeps things tidy
FORMATTED = SLEAP / "formatted"

assert ASSAY in PHASES, f"unknown assay {ASSAY!r}; add it to PHASES"
for d in (ANIMAL, GEOM):
    assert d.is_dir(), f"missing input folder: {d}"
MERGED.mkdir(exist_ok=True)
FORMATTED.mkdir(exist_ok=True)

print(f"{BATCH} / {ASSAY}")
print(f"  animal : {len(list(ANIMAL.glob('*.csv'))):3d} csv")
print(f"  geom   : {len(list(GEOM.glob('*.csv'))):3d} csv")
print(f"  phases : {PHASES[ASSAY] or '(none)'}")
print(f"  output : {FORMATTED}")

## 2. Helpers

`parse_key` is the part that used to be a `split` on underscore position. The animal
code is four alphanumerics and the phase comes from a known set, so both can be matched
directly wherever they sit in the name:

```
E9_B6_SocP-animal_3.004_E9_SIS_B6_D3U7_S1.analysis_locs.csv
                                  ^^^^ ^^
                                  code phase
```

In [ ]:
def parse_key(filename: str, phases: tuple) -> tuple:
    """Return (code, phase) from a SLEAP export filename.

    Matches on content rather than on underscore position, so it is unaffected by
    however many prefix fields the export carries. Raises rather than guessing.
    """
    stem = Path(filename).name
    if phases:
        pat = r"_([A-Z0-9]{4})_(" + "|".join(phases) + r")\."
        hits = re.findall(pat, stem)
        if len(hits) != 1:
            raise ValueError(
                f"expected exactly one <CODE>_<PHASE> in {stem!r}, found {len(hits)}: {hits}"
            )
        return hits[0]
    hits = re.findall(r"_([A-Z0-9]{4})\.", stem)
    if len(hits) != 1:
        raise ValueError(
            f"expected exactly one <CODE> in {stem!r}, found {len(hits)}: {hits}"
        )
    return (hits[0], "")


def index_folder(folder: Path, phases: tuple) -> dict:
    """Map (code, phase) -> path for every CSV in a folder, refusing duplicates."""
    out = {}
    for p in sorted(folder.glob("*.csv")):
        key = parse_key(p.name, phases)
        if key in out:
            raise ValueError(
                f"two files map to {key} in {folder.name}:\n  {out[key].name}\n  {p.name}"
            )
        out[key] = p
    return out


def stem_for(code: str, phase: str) -> str:
    return f"{code}_{phase}" if phase else code


def dlc_header(bodyparts: list) -> list:
    """The three DLCAnalyzer header rows for a set of bodyparts.

    Row 1 reproduces the column_N / column_N_new names the previous script emitted,
    so output stays byte-identical to the files already in formatted/. DLCAnalyzer
    reads rows 2 and 3; row 1 is the scorer row and its contents are not used.
    """
    r1 = ["column_1"]
    for i in range(1, len(bodyparts) + 1):
        r1 += [f"column_{2 * i}", f"column_{2 * i + 1}", f"column_{2 * i + 1}_new"]
    r2 = ["bodyparts"] + [bp for bp in bodyparts for _ in range(3)]
    r3 = ["coords"] + ["x", "y", "likelihood"] * len(bodyparts)
    return [r1, r2, r3]


print("helpers defined")

## 3. Merge `geom` + `animal`

Geometry columns come first, then the animal bodyparts — the order DLCAnalyzer's zone
definitions expect. Pairing is on the parsed key, and anything unmatched on either side
is listed rather than skipped in silence.

In [ ]:
ph = PHASES[ASSAY]
animal_idx = index_folder(ANIMAL, ph)
geom_idx = index_folder(GEOM, ph)

only_animal = sorted(set(animal_idx) - set(geom_idx))
only_geom = sorted(set(geom_idx) - set(animal_idx))
paired = sorted(set(animal_idx) & set(geom_idx))

print(f"paired: {len(paired)}   animal-only: {len(only_animal)}   geom-only: {len(only_geom)}")
for k in only_animal:
    print(f"  !! no geom for   {k}  ({animal_idx[k].name})")
for k in only_geom:
    print(f"  !! no animal for {k}  ({geom_idx[k].name})")

merged_paths, problems = {}, []
for key in paired:
    code, phase = key
    g = pd.read_csv(geom_idx[key])
    a = pd.read_csv(animal_idx[key])
    if len(g) != len(a):
        problems.append(f"{key}: geom has {len(g)} rows, animal has {len(a)} -- skipped")
        continue
    out = MERGED / f"{stem_for(code, phase)}.merged.csv"
    pd.concat([g, a], axis=1).to_csv(out, index=False)
    merged_paths[key] = out

for p in problems:
    print(f"  !! {p}")
print(f"\nmerged {len(merged_paths)} file(s) into {MERGED}")

## 4. Format for DLCAnalyzer

SLEAP's `analysis_locs` export gives `<bodypart>_x, <bodypart>_y` pairs and no
confidence column, so a `likelihood` of 1 is inserted for every bodypart. The number of
bodyparts is read from the header — this is what the old `+20` / `+18` / `+26` per-assay
constants were standing in for.

In [ ]:
def format_one(src: Path, dst: Path) -> dict:
    df = pd.read_csv(src)

    # columns arrive as <bodypart>_x, <bodypart>_y pairs
    if len(df.columns) % 2:
        raise ValueError(f"{src.name}: odd column count {len(df.columns)}; expected x/y pairs")
    bodyparts, seen = [], set()
    for c in df.columns:
        bp, _, axis = c.rpartition("_")
        if axis not in ("x", "y"):
            raise ValueError(f"{src.name}: column {c!r} does not end in _x or _y")
        if bp not in seen:
            seen.add(bp)
            bodyparts.append(bp)
    for bp in bodyparts:
        for axis in ("x", "y"):
            if f"{bp}_{axis}" not in df.columns:
                raise ValueError(f"{src.name}: {bp} is missing its _{axis} column")

    body = pd.DataFrame({"frame": range(len(df))})
    for bp in bodyparts:
        body[f"{bp}_x"] = df[f"{bp}_x"].values
        body[f"{bp}_y"] = df[f"{bp}_y"].values
        body[f"{bp}_likelihood"] = 1

    expected = 1 + 3 * len(bodyparts)
    assert body.shape[1] == expected, f"{src.name}: built {body.shape[1]} cols, expected {expected}"

    # One explicit line terminator for both the hand-written header and the pandas
    # body. Left implicit, the header gets "\n" while pandas writes "\r\n" on
    # Windows, so the file is internally inconsistent -- and 3 bytes different from
    # the existing corpus, which is CRLF throughout.
    dst.parent.mkdir(parents=True, exist_ok=True)
    with open(dst, "w", newline="") as fh:
        for row in dlc_header(bodyparts):
            fh.write(",".join(row) + LINE_TERM)
        body.to_csv(fh, index=False, header=False, lineterminator=LINE_TERM)

    return {"bodyparts": len(bodyparts), "cols": expected, "frames": len(df)}


# Plan every output path first and abort on a collision. The previous version wrote
# 19 files to one name and lost 18 of them without a word.
plan = {}
for key, src in merged_paths.items():
    code, phase = key
    dst = (FORMATTED / phase / f"{stem_for(code, phase)}_formatted.csv") if phase \
          else (FORMATTED / f"{code}_formatted.csv")
    if dst in plan.values():
        raise RuntimeError(f"name collision: {key} and another key both map to {dst}")
    plan[key] = dst

existing = [d for d in plan.values() if d.exists()]
if existing and not OVERWRITE:
    raise RuntimeError(
        f"{len(existing)} output file(s) already exist, e.g. {existing[0].name}.\n"
        "Set OVERWRITE = True in the config cell if replacing them is intended."
    )

rows = []
for key, src in merged_paths.items():
    info = format_one(src, plan[key])
    rows.append({"code": key[0], "phase": key[1], **info, "out": plan[key].name})

report = pd.DataFrame(rows).sort_values(["phase", "code"])
print(f"formatted {len(report)} file(s)\n")
print(report.to_string(index=False))

n_bp = report["bodyparts"].unique()
n_fr = report["frames"].unique()
print(f"\nbodyparts per file: {sorted(n_bp)}" + ("  <-- INCONSISTENT" if len(n_bp) > 1 else ""))
print(f"frames per file:    {sorted(n_fr)}" + ("  <-- differing lengths" if len(n_fr) > 1 else ""))

## 5. Verify

Two checks worth having. The first confirms the written files really are distinct — the
failure this notebook was rewritten for produced one file, not many, and nothing said
so. The second flags any pair of phases whose tracking is identical, which is how a
duplicated recording shows up (see `SISanalyzer/exp9_publication/data/DATA_ISSUES.md`).

In [ ]:
import hashlib
from itertools import combinations

written = sorted(plan.values())
digests = {}
for p in written:
    digests[p] = hashlib.sha256(p.read_bytes()).hexdigest()

dupes = {}
for p, h in digests.items():
    dupes.setdefault(h, []).append(p.name)
collisions = {h: v for h, v in dupes.items() if len(v) > 1}
print(f"{len(written)} file(s) written, {len(dupes)} distinct by content")
if collisions:
    print("  !! byte-identical outputs:")
    for v in collisions.values():
        print(f"     {v}")
else:
    print("  all outputs distinct")

# Per animal, compare the coordinate block between phases. Two exports of the same
# recording agree to ~1e-4 px; two genuine sessions differ by tens of pixels.
if ph:
    print("\nper-animal agreement between phases (mean |diff| in px):")
    by_code = {}
    for (code, phase), dst in plan.items():
        by_code.setdefault(code, {})[phase] = dst
    flagged = []
    for code, d in sorted(by_code.items()):
        for p1, p2 in combinations(sorted(d), 2):
            a = pd.read_csv(d[p1], skiprows=3, header=None).iloc[:, 1:]
            b = pd.read_csv(d[p2], skiprows=3, header=None).iloc[:, 1:]
            n = min(len(a), len(b))
            m = (a.iloc[:n].values - b.iloc[:n].values).__abs__().mean()
            tag = "  <-- SAME RECORDING?" if m < 1 else ""
            if tag:
                flagged.append((code, p1, p2, m))
            print(f"  {code}  {p1} vs {p2}: {m:12.6f}{tag}")
    print()
    if flagged:
        print(f"!! {len(flagged)} phase pair(s) look like the same recording:")
        for code, p1, p2, m in flagged:
            print(f"   {code} {p1}/{p2}  mean |diff| = {m:.2e} px")
        print("   Check the source videos before analysing these.")
    else:
        print("no duplicated recordings detected")